# Modeling, Pipeline, Hyperparameter Tuning & Evaluation

## 1. Import Required Libraries

In [30]:
import pandas as pd              # Used for data manipulation and analysis
import numpy as np               # Used for numerical computations and array operations

from sklearn.model_selection import train_test_split   # Splits data into training and testing sets
from sklearn.model_selection import GridSearchCV       # Performs hyperparameter tuning using cross-validation
from sklearn.model_selection import cross_val_score    # Evaluates model performance using cross-validation

from sklearn.preprocessing import OneHotEncoder        # Converts categorical variables into numerical form
from sklearn.preprocessing import StandardScaler       # Scales numerical features to a standard range

from sklearn.compose import ColumnTransformer           # Applies different preprocessing to different columns
from sklearn.pipeline import Pipeline                  # Creates a machine learning pipeline

from sklearn.linear_model import LinearRegression       # Linear regression model for prediction
from sklearn.ensemble import RandomForestRegressor      # Ensemble model using multiple decision trees
from sklearn.ensemble import GradientBoostingRegressor  # Boosting-based ensemble regression model
from sklearn.metrics import mean_squared_error

## 2. Load Cleaned Dataset (from EDA notebook)

In [15]:
data = pd.read_csv('processed_data.csv')

In [16]:
data.columns

Index(['Drug', 'Demand_Forecast', 'Optimal_Stock_Level',
       'Restocking_Strategy'],
      dtype='object')

# 3. Define Features & Target

In [17]:
x = data.drop(columns=['Optimal_Stock_Level'])
y = data['Optimal_Stock_Level']

# 4. Identify Column Types

In [18]:
numeric_features=x.select_dtypes(include=[np.number]).columns.to_list()
categorical_features=x.select_dtypes(include='object').columns.to_list()

In [19]:
numeric_features

['Demand_Forecast']

In [20]:
categorical_features

['Drug', 'Restocking_Strategy']

# 5. Preprocessing Pipeline

In [21]:
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())   # Scales numerical features to have mean = 0 and standard deviation = 1
])

In [22]:
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(
        drop='first',              # Drops the first category to avoid the dummy variable trap
        handle_unknown='ignore'    # Ignores unseen categories during testing
    ))
])

In [23]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),   # Applies scaling to numerical columns
    ('cat', categorical_transformer, categorical_features)  # Applies one-hot encoding to categorical columns
])

# 6. Train-Test Split

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,        # Uses 20% of the data for testing and 80% for training
    random_state=42       # Ensures reproducible results by fixing the random split
)


# 7. Linear Regression Model

In [25]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),     # Applies preprocessing (scaling + encoding) to the data
    ('model', LinearRegression())        # Linear Regression model for prediction
])

In [26]:
lr_pipeline.fit(X_train, y_train)        # Trains the pipeline on the training data


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Demand_Forecast']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Drug',
                                                   'Restocking_Strategy'])])),
                ('model', LinearRegression())])

In [31]:
y_pred_lr = lr_pipeline.predict(X_test)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))


print("Linear Regression RMSE:", rmse_lr)

Linear Regression RMSE: 2738.236359636495


In [33]:
# Cross-validation
cv_lr = cross_val_score(lr_pipeline, x, y, cv=5, scoring='neg_root_mean_squared_error')
print("Linear Regression CV RMSE:", -cv_lr.mean())

Linear Regression CV RMSE: 2744.6364291612135


# 8. Random Forest Regressor

In [35]:
rf_pipeline = Pipeline(steps=[
('preprocessor', preprocessor),
('model', RandomForestRegressor(random_state=42))
])


rf_pipeline.fit(X_train, y_train)


y_pred_rf = rf_pipeline.predict(X_test)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))


print("Random Forest RMSE:", rmse_rf)


cv_rf = cross_val_score(rf_pipeline, x, y, cv=5, scoring='neg_root_mean_squared_error')
print("Random Forest CV RMSE:", -cv_rf.mean())

Random Forest RMSE: 3164.7781976912156
Random Forest CV RMSE: 3171.7481639015523


# 9. Gradient Boosting Regressor


In [37]:



gb_pipeline = Pipeline(steps=[
('preprocessor', preprocessor),
('model', GradientBoostingRegressor(random_state=42))
])


gb_pipeline.fit(X_train, y_train)


y_pred_gb = gb_pipeline.predict(X_test)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))


print("Gradient Boosting RMSE:", rmse_gb)


cv_gb = cross_val_score(gb_pipeline, x, y, cv=5, scoring='neg_root_mean_squared_error')
print("Gradient Boosting CV RMSE:", -cv_gb.mean())

Gradient Boosting RMSE: 2740.7020081367195
Gradient Boosting CV RMSE: 2745.584004893744


# Production / Final Model: Linear Regression

## Reason for Selection:

- It achieved the lowest RMSE among all evaluated models, indicating better prediction accuracy.
- Linear Regression is simple and computationally efficient, making it suitable for production use.
- The model is easy to interpret and explain to non-technical stakeholders.
- It shows minimal risk of overfitting due to its low model complexity.
- Provides stable and consistent performance on both training and test data.


# 10. Hyperparameter Tuning (Linear Regression)

In [38]:

from sklearn.linear_model import Ridge, Lasso

# ----- Ridge Regression Pipeline -----
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

ridge_param_grid = {
    'model__alpha': [0.1, 1.0, 10.0, 100.0]
}

ridge_grid = GridSearchCV(
    ridge_pipeline,
    ridge_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

ridge_grid.fit(X_train, y_train)

print("Best Ridge Parameters:", ridge_grid.best_params_)

Best Ridge Parameters: {'model__alpha': 100.0}


# 11. Final Model Evaluation (Ridge Regression)


In [40]:
best_lr_model = ridge_grid.best_estimator_

y_pred_final = best_lr_model.predict(X_test)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))

print("Final Tuned Linear Model RMSE:", rmse_final)

cv_final = cross_val_score(
    best_lr_model,
    x, y,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

print("Final Linear Model CV RMSE:", -cv_final.mean())

print("Linear Regression Modeling & Evaluation Completed Successfully")

Final Tuned Linear Model RMSE: 2738.235233945531
Final Linear Model CV RMSE: 2744.6343252682846
Linear Regression Modeling & Evaluation Completed Successfully


In [44]:
lasso_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Lasso(max_iter=5000))
])
lasso_pipeline.fit(X_train, y_train)

y_pred_lasso = lasso_pipeline.predict(X_test)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))

print("Lasso RMSE:", rmse_lasso)
cv_lasso = cross_val_score(
    lasso_pipeline,
    x, y,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

print("Lasso CV RMSE:", -cv_lasso.mean())


Lasso RMSE: 2738.222013505084
Lasso CV RMSE: 2744.5941391649276
